In [1]:
import os
%pwd

'f:\\ML Sushmoy\\Detecting-Thyroid-Cancer Efficientnetb0\\research'

In [2]:
os.chdir("../")
%pwd

'f:\\ML Sushmoy\\Detecting-Thyroid-Cancer Efficientnetb0'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class FeatureSelectionConfig:
    root_dir: Path
    selected_features_path: Path
    base_model_path: Path
    training_data: Path
    params_image_size: list
    params_batch_size: int
    params_classes: int
    params_learning_rate: float
    num_features_to_select: int

In [4]:
from ThyroidCancer.constants import *
from ThyroidCancer.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_feature_selection_config(self) -> FeatureSelectionConfig:
        config = self.config.feature_selection
        prepare_base_model_config = self.config.prepare_base_model
        data_ingestion_config = self.config.data_ingestion
        
        create_directories([config.root_dir])

        feature_selection_config = FeatureSelectionConfig(
            root_dir=Path(config.root_dir),
            selected_features_path=Path(config.selected_features_path),
            base_model_path=Path(prepare_base_model_config.updated_base_model_path),
            training_data=Path(data_ingestion_config.unzip_dir),
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE,
            params_classes=self.params.CLASSES,
            params_learning_rate=self.params.LEARNING_RATE,
            num_features_to_select=self.params.NUM_FEATURES_TO_SELECT
        )

        return feature_selection_config

In [6]:
import tensorflow as tf
import numpy as np
import pickle
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from ThyroidCancer import logger

In [7]:
class FeatureSelection:
    def __init__(self, config: FeatureSelectionConfig):
        self.config = config
    
    def _build_full_model(self):
        """Rebuild the full model architecture (same as in prepare_base_model)"""
        # Load base EfficientNetB0
        base_model = tf.keras.applications.EfficientNetB0(
            input_shape=self.config.params_image_size,
            weights='imagenet',
            include_top=False
        )
        base_model.trainable = False
        
        # Add the same head as in prepare_base_model
        x = tf.keras.layers.GlobalAveragePooling2D(name="avg_pool")(base_model.output)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.5, name="top_dropout")(x)
        predictions = tf.keras.layers.Dense(
            units=self.config.params_classes,
            activation="softmax",
            name="predictions"
        )(x)

        full_model = tf.keras.models.Model(
            inputs=base_model.input,
            outputs=predictions,
            name="EfficientNetB0_transfer"
        )
        
        return full_model
    
    def get_base_model(self):
        """Reconstruct model and load weights"""
        logger.info("Reconstructing model architecture...")
        full_model = self._build_full_model()
        
        # Load weights
        logger.info(f"Loading weights from {self.config.base_model_path}...")
        full_model.load_weights(self.config.base_model_path)
        logger.info("Weights loaded successfully.")
        
        # Create feature extractor (output from avg_pool layer)
        self.feature_extractor = tf.keras.Model(
            inputs=full_model.input,
            outputs=full_model.get_layer('avg_pool').output
        )
        
        logger.info(f"Feature extractor created. Output shape: {self.feature_extractor.output_shape}")

    def generate_data_generator(self):
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=False, # Important for feature extraction alignment
            **dataflow_kwargs
        )
        
        logger.info(f"Data generator created. Found {self.generator.samples} images.")

    def extract_features(self):
        logger.info("Starting feature extraction...")
        
        # Calculate steps to cover all samples
        steps = int(np.ceil(self.generator.samples / self.generator.batch_size))
        
        self.features = self.feature_extractor.predict(self.generator, steps=steps, verbose=1)
        self.labels = self.generator.classes
        
        logger.info(f"Extracted features shape: {self.features.shape}")
        logger.info(f"Labels shape: {self.labels.shape}")
        
    def select_features(self):
        logger.info(f"Starting Feature Selection (RFE) to select {self.config.num_features_to_select} features...")
        
        estimator = RandomForestClassifier(n_estimators=50, n_jobs=-1, random_state=42)
        selector = RFE(estimator, n_features_to_select=self.config.num_features_to_select, step=50)
        
        selector = selector.fit(self.features, self.labels)
        
        selected_indices = np.where(selector.support_)[0]
        logger.info(f"Selected {len(selected_indices)} features.")
        
        # Save selected indices
        with open(self.config.selected_features_path, 'wb') as f:
            pickle.dump(selected_indices, f)
            
        logger.info(f"Selected feature indices saved to {self.config.selected_features_path}")
        return selected_indices

In [8]:
try:
    config = ConfigurationManager()
    feature_selection_config = config.get_feature_selection_config()
    feature_selection = FeatureSelection(config=feature_selection_config)
    
    feature_selection.get_base_model()
    feature_selection.generate_data_generator()
    feature_selection.extract_features()
    feature_selection.select_features()
    
except Exception as e:
    raise e

[2026-01-22 13:22:48,163: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-01-22 13:22:48,166: INFO: common: yaml file: params.yaml loaded successfully]
[2026-01-22 13:22:48,168: INFO: common: created directory at: artifacts]
[2026-01-22 13:22:48,170: INFO: common: created directory at: artifacts/feature_selection]
[2026-01-22 13:22:48,170: INFO: 2455788622: Reconstructing model architecture...]
[2026-01-22 13:22:54,231: INFO: 2455788622: Loading weights from artifacts\prepare_base_model\base_model_updated.h5...]
[2026-01-22 13:22:54,386: INFO: 2455788622: Weights loaded successfully.]
[2026-01-22 13:22:54,394: INFO: 2455788622: Feature extractor created. Output shape: (None, 1280)]
Found 2492 images belonging to 1 classes.
[2026-01-22 13:22:54,493: INFO: 2455788622: Data generator created. Found 2492 images.]
[2026-01-22 13:22:54,493: INFO: 2455788622: Starting feature extraction...]
78/78 [==============================] - 12s 65ms/step
[2026-01-22 13:23:06,192: